In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
批量解析 ACS HTML 文献：
- 用 ACSParser 提取 meta（title / journal / date / abstract）
- 用 parse_paragraphs() 提取正文段落
- 每篇文章输出一个 .txt 文件
"""

import os
from pathlib import Path

DATA_ROOT = Path("Data")
from read.acs import ACSParser  # ✅ 确保你把上面那段 ACSParser 放在 read/acs.py 里


def parse_acs_article(filepath, output_folder):
    """解析单篇 ACS HTML 文件并将 paragraph 写入 txt"""
    try:
        # 1. 初始化解析器
        parser = ACSParser(filepath)

        # 2. 解析元信息（当前 ACSParser.parse_meta 只设置属性，不返回 dict）
        meta_ret = parser.parse_meta()
        if isinstance(meta_ret, dict):
            meta_data = meta_ret
        else:
            meta_data = {
                "title": getattr(parser, "title", ""),
                "journal": getattr(parser, "journal", ""),
                "date": getattr(parser, "date", ""),
                "abstract": getattr(parser, "abstract", ""),
            }

        print(f"📄 Title: {meta_data.get('title', 'N/A')}")
        print(f"📘 Journal: {meta_data.get('journal', 'N/A')}")
        # print(f"🧾 Abstract: {meta_data.get('abstract', 'N/A')}")
        # print(f"📅 Date: {meta_data.get('date', 'N/A')}")

        # 3. 解析段落（ACSParser.parse_paragraphs 返回的是“元素列表”，要自己取 text_content）
        para_elements = parser.parse_paragraphs()  # list of lxml elements
        paragraph_texts = []

        for el in para_elements:
            if hasattr(el, "text_content"):
                txt = el.text_content().strip()
            else:
                txt = str(el).strip()

            if txt:
                paragraph_texts.append(txt)

        print(f"📝 段落数: {len(paragraph_texts)}")

        # 4. 构建输出文件名（去掉后缀，变成安全文件名）
        base_name = os.path.basename(filepath)
        for ext in [".html", ".htm", ".xhtml", ".xml"]:
            if base_name.lower().endswith(ext):
                base_name = base_name[: -len(ext)]
                break

        safe_name = "".join(c for c in base_name if c.isalnum() or c in (" ", "_", "-"))
        output_path = os.path.join(output_folder, f"{safe_name}.txt")

        # 5. 写入到 txt 文件
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(f"Title: {meta_data.get('title', '')}\n")
            f.write(f"Journal: {meta_data.get('journal', '')}\n")
            f.write(f"Date: {meta_data.get('date', '')}\n")
            f.write(f"Abstract: {meta_data.get('abstract', '')}\n\n")
            f.write("Paragraphs:\n")
            for para in paragraph_texts:
                f.write(para + "\n\n")

        print(f"✅ 已保存段落 → {output_path}\n")

        # 如果想顺便调试表格解析，这里可以打开：
        # parser.parse_tables()

    except Exception as e:
        print(f"❌ 解析失败：{filepath}\n错误信息：{e}\n")


def batch_parse_acs_folder(input_folder, output_folder):
    """批量解析整个文件夹下的 ACS HTML 文件，带【自动跳过】功能"""
    os.makedirs(output_folder, exist_ok=True)

    # 1. 筛选文件
    html_files = [
        f for f in os.listdir(input_folder)
        if f.lower().endswith((".html", ".htm", ".xhtml", ".xml"))
    ]

    if not html_files:
        print("⚠️ 未找到 HTML/XML 文件，请检查路径。")
        return

    print(f"🚀 开始批量解析 ACS 文献，共 {len(html_files)} 篇...\n")

    for html_file in html_files:
        # --- 核心改进：预判输出文件路径 ---
        # 这里的逻辑必须和 parse_acs_article 内部生成文件名的逻辑严格对齐
        base_name = html_file
        for ext in [".html", ".htm", ".xhtml", ".xml"]:
            if base_name.lower().endswith(ext):
                base_name = base_name[: -len(ext)]
                break
        
        # 处理安全文件名（保留字母数字、空格、下划线、连字符）
        safe_name = "".join(c for c in base_name if c.isalnum() or c in (" ", "_", "-"))
        output_path = os.path.join(output_folder, f"{safe_name}.txt")

        # --- 自动检索：如果文件已存在且不为空，则跳过 ---
        if os.path.exists(output_path) and os.path.getsize(output_path) > 100:
            # print(f"⏭️  跳过已处理: {safe_name}.txt")
            continue
        # ----------------------------------

        file_path = os.path.join(input_folder, html_file)
        parse_acs_article(file_path, output_folder)

    print("🎯 全部解析完成！")


if __name__ == "__main__":
    # 👉 换成你的 ACS HTML 文件夹路径
    input_folder = DATA_ROOT / "ACS" / "source"   # 📂 输入 HTML/XML 文件夹路径
    output_folder = DATA_ROOT / "ACS" / "txt"     # 📂 输出 TXT 文件夹路径

    batch_parse_acs_folder(input_folder, output_folder)


🚀 开始批量解析 ACS 文献，共 284 篇...

Title: Molecular Modeling for Artificial Metalloenzyme Design and Optimization Molecular Modeling for Artificial Metalloenzyme Design and Optimization Molecular Modeling for Artificial Metalloenzyme Design and Optimization
Journal: 
Date: April 1, 2020 20 January 2020 1 April 2020 21 April 2020 July 17, 2017 July 16, 2019 July 11, 2022 January 28, 2019 February 22, 2019 20 January 2020 1 April 2020 21 April 2020 July 17, 2017 July 16, 2019 July 11, 2022 January 28, 2019 February 22, 2019
📄 Title: Molecular Modeling for Artificial Metalloenzyme Design and Optimization Molecular Modeling for Artificial Metalloenzyme Design and Optimization Molecular Modeling for Artificial Metalloenzyme Design and Optimization
📘 Journal: 
✅ ACS 正文段落数量: 48

📝 段落数: 48
✅ 已保存段落 → D:\FXR\1111-HTML\ACS\TXT\101021_acsaccounts0c00031.txt

Title: Evolution of Anion Relay Chemistry: Construction of Architecturally Complex Natural Products Evolution of Anion Relay Chemistry: Construction

In [ ]:
import os
import pandas as pd

FOLDER = DATA_ROOT / "DOI" / "IOP_DOI_Results"
target_journal = "Matter and Radiation at Extremes"

# 可以按需再加别的编码
CANDIDATE_ENCODINGS = ["utf-8-sig", "utf-8", "gb18030", "latin1"]

for file in os.listdir(FOLDER):
    if not file.lower().endswith(".csv"):
        continue

    path = os.path.join(FOLDER, file)

    df = None
    used_encoding = None

    # 尝试多种编码读入
    for enc in CANDIDATE_ENCODINGS:
        try:
            df = pd.read_csv(path, encoding=enc)
            used_encoding = enc
            break
        except UnicodeDecodeError:
            continue

    if df is None:
        print(f"❌ 无法解码文件（尝试多种编码失败）：{file}")
        continue

    print(f"📂 处理 {file}（使用编码: {used_encoding}）")

    if "Journal" not in df.columns:
        print(f"⚠️ 跳过 {file}：没有 Journal 列")
        continue

    before = len(df)
    df = df[df["Journal"].astype(str).str.strip() != target_journal]
    after = len(df)

    # 原地覆盖写回（统一存成 utf-8-sig）
    df.to_csv(path, index=False, encoding="utf-8-sig")

    print(f"📝 删除 {before - after} 行，剩余 {after} 行")

print("\n🎉 处理完成！所有文件已原地修改。")
